In [ ]:
import os 
from getpass import getpass
from dotenv import load_dotenv
import json

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.base import RunnableSerializable
from langchain_core.messages import ToolMessage
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

load_dotenv()

In [ ]:
# must enter API key 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# below should not be changed
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# you can change this as preferred
os.environ["LANGCHAIN_PROJECT"] = "damar-ananta-langsmith-openai"

In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# For normal accurate responses
llm = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")

In [ ]:
""" 
1. Apa itu Agent Executor?
Agent pada dasarnya adalah logic yang:
1. Memanggil LLM.
2. Memproses output LLM.
3. Jika LLM meminta Tool, menjalankan Tool tersebut.
4. Mengirim hasil Tool kembali ke LLM.
5. Mengulangi proses sampai mendapatkan jawaban akhir.

Menggunakan pola ReAct:
- Reason → LLM menentukan langkah.
- Action → LLM memilih Tool.
- Observation → Tool memberikan hasil.
- Kembali ke Reason jika diperlukan.

Agent bukan hanya LLM. Sebagian besar logic Agent adalah kode yang menjalankan LLM secara berulang dan memproses output-nya.
"""

# Tools

In [ ]:
@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

# Define the multiply tool
@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

# Define the exponentiate tool
@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

In [ ]:
add

In [ ]:
print(f"{add.name=}\n{add.description=}")

In [ ]:
add.args_schema.model_json_schema()

In [ ]:
exponentiate.args_schema.model_json_schema()

In [ ]:
llm_output_string = "{\"x\": 5, \"y\": 2}"  # this is the output from the LLM
llm_output_dict = json.loads(llm_output_string)  # load as dictionary
llm_output_dict

In [ ]:
exponentiate.func(**llm_output_dict)

# Creating an Agent

In [ ]:
""" 
User
  ↓
LLM
  ↓
Memilih Tool
  ↓
Tool dijalankan
  ↓
Hasil Tool
  ↓
LLM
  ↓
Jawaban

    Siapa yang mengatur proses berulang tersebut?

Jawabannya adalah Agent Execution Loop. Pada LangChain versi lama, proses ini dikendalikan oleh AgentExecutor. Pada LangChain 1.5.2, loop tersebut sudah menjadi bagian dari runtime Agent yang dibuat menggunakan create_agent, yang menggunakan LangGraph di bawahnya.


Misalkan kita membuat Agent yang memiliki Tool:
@tool
def add(x: int, y: int) -> int:
    """Add two numbers."""
    return x + y

User bertanya:
    What is 10 + 10?

LLM tidak langsung menghitung dan memberikan jawaban. Ia dapat menghasilkan Tool Call seperti:
{
    "name": "add",
    "args": {
        "x": 10,
        "y": 10
    }
}

Output ini merupakan instruksi untuk menjalankan Tool, bukan final answer.

Kemudian sistem perlu:
1. Membaca Tool Call.
2. Menemukan Tool bernama add.
3. Memberikan argument x=10, y=10.
4. Menjalankan fungsi.
5. Mengambil hasil 20.
6. Mengirim hasil tersebut kembali kepada LLM.
7. Meminta LLM menentukan langkah selanjutnya.
Proses inilah yang disebut Agent Execution Loop.
"""

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a helpful assistant. When answering a user's question "
        "you should first use one of the tools provided. After using a "
        "tool the tool output will be provided in the "
        "'scratchpad' below. If you have an answer in the "
        "scratchpad you should not use any more tools and "
        "instead answer directly to the user."
    )),
    MessagesPlaceholder(variable_name="chat_history"), #Tempat untuk menyisipkan percakapan sebelumnya.
    ("human", "{input}"), #Tempat pertanyaan User dimasukkan.
    MessagesPlaceholder(variable_name="agent_scratchpad"), #Tempat Agent menyimpan proses Tool Calling yang telah dilakukan.
])

""" 
Notebook menjelaskan bahwa scratchpad berisi message yang bergantian:
- AIMessage → berisi Tool Call.
- ToolMessage → berisi hasil eksekusi Tool.
Kedua message tersebut terhubung melalui tool_call_id. ool_call_id digunakan untuk menghubungkan Tool Call dengan Tool Result. Ini sangat penting ketika terdapat beberapa Tool Calls, termasuk ketika beberapa Tool dipanggil secara paralel.

AIMessage
Tool Call ID: call_123
Tool: add(x=10, y=10)

ToolMessage
Tool Call ID: call_123
Result: 20
"""

In [ ]:
tools = [add, subtract, multiply, exponentiate]

# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"], #Mengambil pertanyaan User.
        "chat_history": lambda x: x["chat_history"], #Mengambil chat history.
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", []) #Kalau key agent_scratchpad belum tersedia, gunakan list kosong.
    }
    | prompt # Dictionary yang tadi sudah diproses dimasukkan ke Prompt. Prompt akan menghasilkan messages yang akan dikirim ke LLM.
    | llm.bind_tools(tools, tool_choice="any") # Memberikan Tools kepada LLM. tool_choice -> LLM wajib memilih Tool dan membuat Tool Call. LLM tidak boleh langsung memberikan jawaban natural language pada tahap ini.
)

In [ ]:
tool_call = agent.invoke({"input": "What is 10 + 10", "chat_history": []})
tool_call

""" 
Agent belum menjalankan fungsi add(). LLM hanya menghasilkan permintaan Tool Call.
"""

In [ ]:
# Menunjukkan bahwa Tool Call-nya
tool_call.tool_calls


""" 
[
    {
        "name": "add",
        "args": {
            "x": 10,
            "y": 10
        },
        "id": "call_...",
        "type": "tool_call"
    }
]

Apa arti output tersebut?
LLM memutuskan:
- Tool yang digunakan: add
- Argument x: 10
- Argument y: 10
- Tool Call ID: ID unik untuk request tersebut
Jadi, agent.invoke() pada kode notebook ini hanya menghasilkan keputusan Tool Call.

Belum menjalankan Tool.
"""

In [ ]:
""" 
Menjalankan Tool secara manual
- Notebook menjelaskan bahwa untuk menjalankan Tool, ada dua langkah.
"""

# 1. Mapping nama Tool ke fungsi
name2tool = {tool.name: tool.func for tool in tools}

""" 
{
    "add": add_function,
    "subtract": subtract_function,
    "multiply": multiply_function,
    "exponentiate": exponentiate_function
}
"""

In [ ]:
# 2. Menjalankan fungsi Tool

tool_exec_content = name2tool[tool_call.tool_calls[0]["name"]]( #tool_call.tool_calls[0]["name"] -> Mengambil nama Tool
    **tool_call.tool_calls[0]["args"] #tool_call.tool_calls[0]["args"] -> Mengambil argument
    # Menggunakan ** -> Mengubah dictionary menjadi keyword arguments. Sehingga keseluruhan kode setara dengan: add(x=10, y=10)
)
tool_exec_content

In [ ]:
tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_exec_content}", # Berisi hasil eksekusi Tool.
    tool_call_id=tool_call.tool_calls[0]["id"] # Menghubungkan Tool Result dengan Tool Call sebelumnya.
)

out = agent.invoke({
    "input": "What is 10 + 10",
    "chat_history": [],
    "agent_scratchpad": [tool_call, tool_exec]
})

""" 
Isi scratchpad berisi: 
[
    tool_call,
    tool_exec
]

Artinya LLM diberikan:
1. Tool Call yang sebelumnya dilakukan.
2. Hasil Tool tersebut.

Secara konsep, LLM menerima konteks:
User:
What is 10 + 10?

AI:
I called the add tool with x=10 and y=10.

Tool:
The add tool returned 20.
"""
out

In [ ]:
# Opsi 1 — tool_choice="auto"

agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="auto") # Dengan auto, LLM boleh: Menggunakan Tool, Memberikan final answer secara langsung. Jadi setelah mendapatkan hasil Tool, LLM dapat memutuskan: Saya sudah mendapatkan jawabannya, tidak perlu Tool lagi.
)

""" 
Masalah tool_choice="any". Ketika Agent dipanggil kembali:
  out = agent.invoke(...)  

LLM malah kembali meminta Tool add. Kenapa? Karena sebelumnya kita mengatur:
    llm.bind_tools(
    tools,
    tool_choice="any"
)

any memaksa LLM untuk selalu menggunakan Tool. Walaupun LLM sudah mendapatkan hasil 20, ia tetap diwajibkan menghasilkan Tool Call. Jadi bisa terjadi:

LLM
 ↓
add(10, 10)
 ↓
20
 ↓
LLM
 ↓
add(10, 10) lagi
 ↓
20
 ↓
LLM
 ↓
add(10, 10) lagi
"""

In [ ]:
tool_call = agent.invoke({"input": "What is 10 + 10", "chat_history": []})
tool_call

In [ ]:
tool_output = name2tool[tool_call.tool_calls[0]["name"]](
    **tool_call.tool_calls[0]["args"]
)

tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_output}",
    tool_call_id=tool_call.tool_calls[0]["id"]
)

out = agent.invoke({
    "input": "What is 10 + 10",
    "chat_history": [],
    "agent_scratchpad": [tool_call, tool_exec]
})
out

In [ ]:
# Opsi kedua: membuat final_answer Tool

@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`.
    """
    return {"answer": answer, "tools_used": tools_used}

""" 
Tool ini tidak melakukan perhitungan. Tujuannya adalah membuat format final answer dan memberi sinyal kepada execution loop bahwa Agent sudah selesai.

final_answer(
    answer="10 + 10 equals 20.",
    tools_used=["functions.add"]
)

{
    "answer": "10 + 10 equals 20.",
    "tools_used": ["functions.add"]
}
Jawaban yang akan diberikan kepada User.

tools_used
Daftar Tools yang digunakan selama proses.

Kenapa notebook memilih opsi kedua?
Notebook menjelaskan beberapa alasan:
1. Mengurangi kemungkinan LLM langsung memberikan output melalui content ketika kita mengharapkan Tool Call.
2. Memberikan format output yang konsisten.
3. Memungkinkan structured output.
4. Bisa memberikan metadata seperti daftar Tools yang digunakan.
"""

In [ ]:
# Menambahkan final_answer ke Tools

tools = [final_answer, add, subtract, multiply, exponentiate]

# we need to update our name2tool mapping too
name2tool = {tool.name: tool.func for tool in tools}

agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
)

In [ ]:
tool_call = agent.invoke({"input": "What is 10 + 10", "chat_history": []})
tool_call.tool_calls

In [ ]:
tool_out = name2tool[tool_call.tool_calls[0]["name"]](
    **tool_call.tool_calls[0]["args"]
)

tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_out}",
    tool_call_id=tool_call.tool_calls[0]["id"]
)

out = agent.invoke({
    "input": "What is 10 + 10",
    "chat_history": [],
    "agent_scratchpad": [tool_call, tool_exec]
})
out

In [ ]:
out.tool_calls

In [ ]:
out.tool_calls[0]["args"]

# Building a Custom Agent Execution Loop

In [ ]:
""" 
Sampai sini kita sudah melakukan semuanya secara manual:
1. Memanggil Agent.
2. Mendapatkan Tool Call.
3. Menjalankan Tool.
4. Membuat ToolMessage.
5. Memasukkan hasil ke Scratchpad.
6. Memanggil Agent kembali.
7. Memeriksa apakah final answer sudah diperoleh.
"""

In [ ]:
class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3): #Saat membuat object, kita menentukan maksimal berapa kali Agent boleh melakukan iteration.
        self.chat_history = [] #Menyimpan history percakapan.
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = ( #Agent yang kita buat sebelumnya dipindahkan ke dalam class. Jadi, object CustomAgentExecutor memiliki Agent yang siap dipanggil.
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    def invoke(self, input: str) -> dict: #Method menerima pertanyaan User.
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0 #Menghitung jumlah iteration Agent.
        agent_scratchpad = [] #Menyimpan Tool Call dan Tool Result dari task saat ini.
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            tool_call = self.agent.invoke({ #Memanggil Agent
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            })
            """ 
            Setiap iteration, Agent mendapatkan:
            - Pertanyaan User.
            - History conversation.
            - Scratchpad dari Tool Call sebelumnya.
            Jadi Agent bisa melihat hasil yang sudah diperoleh sebelumnya.
            """
            # Menambahkan Tool Call ke Scratchpad
            agent_scratchpad.append(tool_call)

            # Mengambil informasi Tool Call
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_calls[0]["id"]
            tool_out = name2tool[tool_name](**tool_args) # Menjalankan Tool
            # add the tool output to the agent scratchpad
            tool_exec = ToolMessage( #Membuat ToolMessage
                content=f"{tool_out}",
                tool_call_id=tool_call_id
            )
            agent_scratchpad.append(tool_exec) #Menambahkan Tool Result ke Scratchpad
            # add a print so we can see intermediate steps
            print(f"{count}: {tool_name}({tool_args})") #Tujuannya untuk melihat Tool yang sedang digunakan Agent.
            count += 1
            # if the tool call is the final answer tool, we stop
            if tool_name == "final_answer": #Jika Agent memilih final_answer, loop dihentikan.
                break
        # add the final output to the chat history
        final_answer = tool_out["answer"]
        self.chat_history.extend([ #Menambahkan ke Chat History
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])
        # return the final answer in dict form
        return json.dumps(tool_out)

In [ ]:
# Membuat dan menjalankan Agent Executor
agent_executor = CustomAgentExecutor()

In [ ]:
agent_executor.invoke(input="What is 10 + 10")